# 🔐 OpenSSL Cryptographic Operations Tutorial

This notebook demonstrates various cryptographic operations using Python's `cryptography` library, which wraps OpenSSL.

## Topics Covered:
1. **RSA Key Pair Generation** - Create public/private key pairs
2. **SHA-512 Hashing** - Hash data securely
3. **X.509 Certificates** - Generate self-signed certificates
4. **Digital Signatures** - Sign and verify messages

---

## 📦 Installation

First, let's install the required cryptography library:

In [ ]:
!pip install cryptography

## 📚 Import Required Libraries

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.backends import default_backend
from cryptography import x509
from cryptography.x509.oid import NameOID
import datetime
import base64

---

## 1️⃣ RSA Key Pair Generation

**What is RSA?**
RSA is an asymmetric cryptographic algorithm that uses a pair of keys:
- **Public Key**: Can be shared with anyone and is used for encryption or signature verification
- **Private Key**: Must be kept secret and is used for decryption or signing

**Key Size Considerations:**
- **2048-bit**: Standard security, faster operations
- **4096-bit**: Enhanced security, slower operations

Let's generate an RSA key pair:

In [ ]:
# Generate a private key with 2048-bit key size
# You can change this to 3072 or 4096 for stronger keys
KEY_SIZE = 2048

print(f"Generating {KEY_SIZE}-bit RSA key pair...")

# Generate the private key
private_key = rsa.generate_private_key(
    public_exponent=65537,  # Standard value for RSA
    key_size=KEY_SIZE,
    backend=default_backend()
)

# Derive the public key from the private key
public_key = private_key.public_key()

print("✓ Key pair generated successfully!")
print(f"\nKey size: {KEY_SIZE} bits")
print(f"Public exponent: 65537")

### Export Keys to PEM Format

PEM (Privacy Enhanced Mail) is a standard format for storing cryptographic keys:

In [ ]:
# Serialize private key to PEM format
private_pem = private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()  # No password protection
)

# Serialize public key to PEM format
public_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)

print("PRIVATE KEY:")
print("=" * 60)
print(private_pem.decode('utf-8'))

print("\nPUBLIC KEY:")
print("=" * 60)
print(public_pem.decode('utf-8'))

---

## 2️⃣ SHA-512 Hashing

**What is SHA-512?**
SHA-512 is a cryptographic hash function that:
- Produces a 512-bit (64-byte) hash value
- Is one-way: you cannot reverse the hash to get the original data
- Is deterministic: same input always produces the same hash
- Is collision-resistant: very difficult to find two inputs with the same hash

**Common Uses:**
- Password storage
- File integrity verification
- Digital signatures

Let's hash some text:

In [ ]:
# Text to hash - try changing this!
text = "Hello, OpenSSL!"

print(f"Original text: {text}")
print(f"Text length: {len(text)} characters")

# Create a hash object
digest = hashes.Hash(hashes.SHA512(), backend=default_backend())

# Update with the text (must be bytes)
digest.update(text.encode('utf-8'))

# Finalize and get the hash
hash_bytes = digest.finalize()

# Convert to hexadecimal for display
hash_hex = hash_bytes.hex()

print(f"\nSHA-512 Hash:")
print("=" * 60)
print(hash_hex)
print(f"\nHash length: {len(hash_hex)} hex characters ({len(hash_bytes)} bytes)")

### 🧪 Experiment: Hash Properties

Let's demonstrate key properties of cryptographic hashes:

In [ ]:
def sha512_hash(text):
    """Helper function to compute SHA-512 hash"""
    digest = hashes.Hash(hashes.SHA512(), backend=default_backend())
    digest.update(text.encode('utf-8'))
    return digest.finalize().hex()

# Property 1: Deterministic - same input produces same hash
text1 = "Hello"
hash1a = sha512_hash(text1)
hash1b = sha512_hash(text1)

print("Property 1: Deterministic")
print(f"Hash of '{text1}' (first):  {hash1a[:32]}...")
print(f"Hash of '{text1}' (second): {hash1b[:32]}...")
print(f"Are they equal? {hash1a == hash1b}")

# Property 2: Avalanche Effect - small change drastically changes hash
text2a = "Hello"
text2b = "hello"  # Just changed capitalization
hash2a = sha512_hash(text2a)
hash2b = sha512_hash(text2b)

print(f"\nProperty 2: Avalanche Effect")
print(f"Hash of '{text2a}': {hash2a[:32]}...")
print(f"Hash of '{text2b}': {hash2b[:32]}...")
print(f"Are they equal? {hash2a == hash2b}")

# Property 3: Fixed output size - regardless of input size
short = "Hi"
long = "This is a much longer string with more characters" * 100

hash_short = sha512_hash(short)
hash_long = sha512_hash(long)

print(f"\nProperty 3: Fixed Output Size")
print(f"Input '{short}' ({len(short)} chars) -> Hash length: {len(hash_short)} chars")
print(f"Input text ({len(long)} chars) -> Hash length: {len(hash_long)} chars")

---

## 3️⃣ X.509 Certificate Creation

**What is an X.509 Certificate?**
An X.509 certificate is a digital document that:
- Contains a public key
- Contains identity information (Common Name, Organization, etc.)
- Is digitally signed to verify authenticity
- Has a validity period (start and end date)

**Self-Signed Certificates:**
- Signed by the same entity that it certifies
- Useful for development and testing
- Not trusted by browsers by default
- Production systems should use certificates from trusted Certificate Authorities (CAs)

Let's create a self-signed certificate:

In [ ]:
# Certificate parameters - feel free to modify these!
COMMON_NAME = "localhost"
COUNTRY = "US"
STATE = "California"
LOCALITY = "San Francisco"
ORGANIZATION = "My Organization"
VALIDITY_DAYS = 365

print(f"Creating self-signed certificate for: {COMMON_NAME}")
print(f"Validity period: {VALIDITY_DAYS} days")

# Generate a new private key for the certificate
cert_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
    backend=default_backend()
)

# Create subject and issuer (same for self-signed)
subject = issuer = x509.Name([
    x509.NameAttribute(NameOID.COUNTRY_NAME, COUNTRY),
    x509.NameAttribute(NameOID.STATE_OR_PROVINCE_NAME, STATE),
    x509.NameAttribute(NameOID.LOCALITY_NAME, LOCALITY),
    x509.NameAttribute(NameOID.ORGANIZATION_NAME, ORGANIZATION),
    x509.NameAttribute(NameOID.COMMON_NAME, COMMON_NAME),
])

# Build the certificate
cert = x509.CertificateBuilder().subject_name(
    subject
).issuer_name(
    issuer
).public_key(
    cert_private_key.public_key()
).serial_number(
    x509.random_serial_number()
).not_valid_before(
    datetime.datetime.utcnow()
).not_valid_after(
    datetime.datetime.utcnow() + datetime.timedelta(days=VALIDITY_DAYS)
).add_extension(
    x509.SubjectAlternativeName([
        x509.DNSName(COMMON_NAME),
    ]),
    critical=False,
).sign(cert_private_key, hashes.SHA512(), default_backend())

print("\n✓ Certificate created successfully!")

# Display certificate details
print(f"\nCertificate Details:")
print("=" * 60)
print(f"Subject: {cert.subject.rfc4514_string()}")
print(f"Issuer: {cert.issuer.rfc4514_string()}")
print(f"Serial Number: {cert.serial_number}")
print(f"Valid From: {cert.not_valid_before}")
print(f"Valid Until: {cert.not_valid_after}")
print(f"Signature Algorithm: {cert.signature_algorithm_oid._name}")

### Export Certificate to PEM Format

In [ ]:
# Serialize certificate to PEM format
cert_pem = cert.public_bytes(serialization.Encoding.PEM)

print("CERTIFICATE:")
print("=" * 60)
print(cert_pem.decode('utf-8'))

# Also export the private key
cert_private_pem = cert_private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

print("\nCERTIFICATE PRIVATE KEY:")
print("=" * 60)
print(cert_private_pem.decode('utf-8'))

---

## 4️⃣ Digital Signatures

**What is a Digital Signature?**
A digital signature:
- Proves that a message was created by the holder of a private key
- Ensures the message hasn't been altered (integrity)
- Provides non-repudiation (sender cannot deny sending)

**How it Works:**
1. **Signing**: Hash the message, then encrypt the hash with the private key
2. **Verification**: Hash the message, decrypt the signature with the public key, compare hashes

**Signature Scheme:**
We'll use RSA-PSS (Probabilistic Signature Scheme) with SHA-512, which is more secure than PKCS#1 v1.5.

Let's sign a message:

In [ ]:
# Message to sign - try changing this!
message = "This is a secure message that needs to be signed."

print(f"Original message: {message}")
print(f"\nSigning message with private key...")

# Sign the message using the private key from earlier
signature = private_key.sign(
    message.encode('utf-8'),
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA512()),  # Mask Generation Function
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA512()
)

# Encode signature to base64 for easy display/transmission
signature_b64 = base64.b64encode(signature).decode('utf-8')

print("\n✓ Message signed successfully!")
print(f"\nSignature (Base64):")
print("=" * 60)
print(signature_b64)
print(f"\nSignature length: {len(signature)} bytes")

### Verify the Digital Signature

Now let's verify that the signature is valid:

In [ ]:
print(f"Verifying signature with public key...")

try:
    # Verify the signature using the public key
    public_key.verify(
        signature,
        message.encode('utf-8'),
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA512()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA512()
    )
    print("\n✓ Signature is VALID!")
    print("The message is authentic and has not been tampered with.")
except Exception as e:
    print("\n✗ Signature is INVALID!")
    print(f"Error: {e}")

### 🧪 Experiment: Tampered Message

Let's see what happens when we try to verify a signature with a modified message:

In [ ]:
# Tampered message (slightly different)
tampered_message = "This is a secure message that needs to be signed!"

print(f"Original message:  {message}")
print(f"Tampered message: {tampered_message}")
print(f"\nVerifying signature with tampered message...")

try:
    public_key.verify(
        signature,
        tampered_message.encode('utf-8'),
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA512()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA512()
    )
    print("\n✓ Signature is VALID (This shouldn't happen!)")
except Exception as e:
    print("\n✗ Signature is INVALID!")
    print("As expected, the signature does not match the tampered message.")
    print(f"This proves the message integrity protection!")

---

## 🛡️ Security Best Practices

### Key Management
1. **Never share private keys** - They should remain secret at all times
2. **Use strong key sizes** - Minimum 2048-bit for RSA, 4096-bit recommended
3. **Store keys securely** - Use hardware security modules (HSMs) or encrypted storage
4. **Rotate keys regularly** - Change keys periodically, especially after potential compromise
5. **Use password protection** - Encrypt private keys with strong passphrases

### Hashing
1. **Use SHA-256 or SHA-512** - Avoid older algorithms like MD5 or SHA-1
2. **Add salt for passwords** - Prevent rainbow table attacks
3. **Use key derivation functions** - PBKDF2, bcrypt, or Argon2 for password hashing

### Certificates
1. **Use trusted CAs in production** - Don't use self-signed certificates for public services
2. **Validate certificate chains** - Ensure certificates are properly signed
3. **Check expiration dates** - Renew certificates before they expire
4. **Enable certificate pinning** - For mobile apps, pin to specific certificates

### Digital Signatures
1. **Use RSA-PSS or ECDSA** - More secure than older schemes
2. **Verify signatures before trusting** - Always validate signatures on received data
3. **Include timestamps** - Prevent replay attacks
4. **Use appropriate hash algorithms** - SHA-256 or SHA-512

### General
1. **Keep libraries updated** - Security patches are released regularly
2. **Use HTTPS/TLS** - Encrypt all network communications
3. **Implement rate limiting** - Prevent brute force attacks
4. **Log security events** - Monitor for suspicious activity
5. **Follow principle of least privilege** - Only grant necessary permissions

---

## 📝 Summary

In this notebook, you learned:

✅ How to generate RSA key pairs for asymmetric cryptography

✅ How to use SHA-512 for secure hashing of data

✅ How to create self-signed X.509 certificates

✅ How to create and verify digital signatures

✅ Best practices for cryptographic operations

### 🚀 Next Steps

- Try the Flask web application for a user-friendly interface
- Experiment with different key sizes and see the performance impact
- Explore symmetric encryption with AES
- Learn about elliptic curve cryptography (ECC)
- Study TLS/SSL protocols and how they use these primitives

### 📚 Additional Resources

- [Cryptography Library Documentation](https://cryptography.io/)
- [OpenSSL Documentation](https://www.openssl.org/docs/)
- [NIST Cryptographic Standards](https://csrc.nist.gov/)
- [OWASP Cryptographic Storage Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Cryptographic_Storage_Cheat_Sheet.html)

---

**Remember:** Cryptography is complex! Always consult with security experts when implementing cryptographic systems for production use.